# Risk-Return Optimization for S&P 500 Sector Allocation

This notebook demonstrates portfolio optimization techniques using the Markowitz framework applied to S&P 500 sector ETFs. We'll explore how to create optimal portfolios based on historical returns data and use this to inform algorithmic trading strategies.

Key concepts covered:
1. Modern Portfolio Theory (MPT) implementation
2. Efficient frontier calculation
3. Sharpe ratio optimization
4. Visualization of risk-return characteristics
5. Portfolio rebalancing framework

## Setup and Data Collection

First, let's import the necessary libraries and fetch the price data for S&P 500 sector ETFs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import yfinance as yf

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

# Set random seed for reproducibility
np.random.seed(42)

Let's define and download data for all the S&P 500 sector ETFs.

In [ ]:
# Define S&P 500 sector ETFs
sectors = {
    'XLF': 'Financials',
    'XLK': 'Technology',
    'XLV': 'Healthcare',
    'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',
    'XLI': 'Industrials',
    'XLU': 'Utilities',
    'XLB': 'Materials',
    'XLE': 'Energy',
    'XLRE': 'Real Estate',
    'XLC': 'Communication Services'
}

# Fetch historical data (last 3 years)
data = yf.download(list(sectors.keys()), start='2022-01-01')

# Extract adjusted close prices
prices = data['Adj Close']

# Display the first few rows
print(f"Data period: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Number of trading days: {len(prices)}")
prices.head()

Let's visualize the price movements of these sector ETFs to get a feel for their relationships.

In [ ]:
# Calculate normalized prices (starting from 100)
normalized_prices = prices.div(prices.iloc[0]).mul(100)

# Plot normalized prices
plt.figure(figsize=(14, 8))
for col in normalized_prices.columns:
    plt.plot(normalized_prices.index, normalized_prices[col], label=f"{col}: {sectors[col]}")
plt.title('S&P 500 Sector ETF Performance (Normalized to 100)')
plt.xlabel('Date')
plt.ylabel('Normalized Price')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Calculating Returns and Risk Metrics

Now, let's calculate the daily returns and analyze the risk-return characteristics of each sector.

In [ ]:
# Calculate daily returns
returns = prices.pct_change().dropna()

# Calculate annualized mean returns and covariance matrix
mean_returns = returns.mean() * 252  # Annualized by multiplying by approximate trading days in a year
cov_matrix = returns.cov() * 252     # Annualized covariance matrix

# Calculate annualized standard deviation (volatility)
volatility = np.sqrt(np.diag(cov_matrix))

# Create a DataFrame with risk-return metrics
risk_return_metrics = pd.DataFrame({
    'Sector': [sectors[ticker] for ticker in returns.columns],
    'Annual Return': mean_returns,
    'Annual Volatility': volatility,
    'Sharpe Ratio': mean_returns / volatility  # Assuming risk-free rate of 0 for simplicity
}, index=returns.columns)

# Display risk-return metrics sorted by Sharpe ratio
display(risk_return_metrics.sort_values('Sharpe Ratio', ascending=False))

In [ ]:
# Plot risk vs. return for each sector
plt.figure(figsize=(12, 8))
plt.scatter(risk_return_metrics['Annual Volatility'], risk_return_metrics['Annual Return'], 
            s=100, alpha=0.8)

# Add labels for each sector
for i, ticker in enumerate(risk_return_metrics.index):
    plt.annotate(f"{ticker}: {sectors[ticker]}", 
                 (risk_return_metrics['Annual Volatility'][i], risk_return_metrics['Annual Return'][i]),
                 textcoords="offset points", xytext=(5,5), ha='left')

plt.title('Risk-Return Profile of S&P 500 Sectors')
plt.xlabel('Annual Volatility (Risk)')
plt.ylabel('Annual Return')
plt.grid(True)
plt.tight_layout()
plt.show()

Let's also visualize the correlation matrix to understand the relationships between different sectors.

In [ ]:
# Calculate correlation matrix
correlation_matrix = returns.corr()

# Rename columns/index to sector names for better visualization
correlation_matrix_labeled = correlation_matrix.copy()
correlation_matrix_labeled.columns = [f"{ticker}\n{sectors[ticker]}" for ticker in correlation_matrix.columns]
correlation_matrix_labeled.index = [f"{ticker}\n{sectors[ticker]}" for ticker in correlation_matrix.index]

# Plot correlation matrix heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix_labeled, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0,
            linewidths=.5, fmt='.2f', cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of S&P 500 Sector ETFs', fontsize=16)
plt.tight_layout()
plt.show()

## Modern Portfolio Theory Implementation

Now, let's implement Modern Portfolio Theory (MPT) to find the efficient frontier and the optimal portfolio allocation.

In [ ]:
# Define functions for portfolio optimization

def portfolio_performance(weights, mean_returns, cov_matrix):
    """Calculate portfolio return and volatility"""
    portfolio_return = np.sum(mean_returns * weights)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return portfolio_return, portfolio_volatility

def portfolio_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.02):
    """Calculate Sharpe ratio for a portfolio"""
    portfolio_return, portfolio_volatility = portfolio_performance(weights, mean_returns, cov_matrix)
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
    return sharpe_ratio

def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.02):
    """Negative Sharpe ratio for minimization"""
    return -portfolio_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate)

def get_min_volatility_portfolio(mean_returns, cov_matrix):
    """Find the minimum volatility portfolio"""
    num_assets = len(mean_returns)
    args = (mean_returns, cov_matrix)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for asset in range(num_assets))
    
    # Starting with equal weights
    initial_weights = np.array([1.0/num_assets] * num_assets)
    
    # Minimize the volatility
    def min_volatility_objective(weights, mean_returns, cov_matrix):
        return portfolio_performance(weights, mean_returns, cov_matrix)[1]
    
    result = minimize(min_volatility_objective, initial_weights, args=args,
                    method='SLSQP', bounds=bounds, constraints=constraints)
    
    min_vol_weights = result['x']
    min_vol_return, min_vol_volatility = portfolio_performance(min_vol_weights, mean_returns, cov_matrix)
    
    return {
        'weights': min_vol_weights,
        'return': min_vol_return,
        'volatility': min_vol_volatility,
        'sharpe': portfolio_sharpe_ratio(min_vol_weights, mean_returns, cov_matrix)
    }

def get_max_sharpe_portfolio(mean_returns, cov_matrix, risk_free_rate=0.02):
    """Find the portfolio with maximum Sharpe ratio"""
    num_assets = len(mean_returns)
    args = (mean_returns, cov_matrix, risk_free_rate)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for asset in range(num_assets))
    
    # Starting with equal weights
    initial_weights = np.array([1.0/num_assets] * num_assets)
    
    # Maximize Sharpe ratio (minimize negative Sharpe)
    result = minimize(negative_sharpe_ratio, initial_weights, args=args,
                    method='SLSQP', bounds=bounds, constraints=constraints)
    
    max_sharpe_weights = result['x']
    max_sharpe_return, max_sharpe_volatility = portfolio_performance(max_sharpe_weights, mean_returns, cov_matrix)
    
    return {
        'weights': max_sharpe_weights,
        'return': max_sharpe_return,
        'volatility': max_sharpe_volatility,
        'sharpe': portfolio_sharpe_ratio(max_sharpe_weights, mean_returns, cov_matrix, risk_free_rate)
    }

def get_efficient_frontier(mean_returns, cov_matrix, returns_range=None, points=100):
    """Calculate the efficient frontier portfolios for a range of returns"""
    min_vol_port = get_min_volatility_portfolio(mean_returns, cov_matrix)
    
    if returns_range is None:
        # Get return range from minimum volatility to maximum return individual asset
        min_return = min_vol_port['return']
        max_return = np.max(mean_returns)
        returns_range = np.linspace(min_return, max_return, points)
    
    efficient_portfolios = []
    
    for target_return in returns_range:
        efficient_portfolio = optimize_for_target_return(mean_returns, cov_matrix, target_return)
        efficient_portfolios.append(efficient_portfolio)
    
    return efficient_portfolios

def optimize_for_target_return(mean_returns, cov_matrix, target_return):
    """Find the portfolio with minimum volatility for a target return"""
    num_assets = len(mean_returns)
    args = (mean_returns, cov_matrix)
    
    # Constraints: weights sum to 1 and portfolio return equals target
    constraints = [
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
        {'type': 'eq', 'fun': lambda x: portfolio_performance(x, mean_returns, cov_matrix)[0] - target_return}
    ]
    
    bounds = tuple((0, 1) for asset in range(num_assets))
    
    # Starting with equal weights
    initial_weights = np.array([1.0/num_assets] * num_assets)
    
    # Minimize volatility for the target return
    def target_return_objective(weights, mean_returns, cov_matrix):
        return portfolio_performance(weights, mean_returns, cov_matrix)[1]
    
    try:
        result = minimize(target_return_objective, initial_weights, args=args,
                        method='SLSQP', bounds=bounds, constraints=constraints)
        
        if not result['success']:
            # If optimization fails, return None
            return None
        
        weights = result['x']
        return_val, volatility = portfolio_performance(weights, mean_returns, cov_matrix)
        sharpe = portfolio_sharpe_ratio(weights, mean_returns, cov_matrix)
        
        return {
            'weights': weights,
            'return': return_val,
            'volatility': volatility,
            'sharpe': sharpe
        }
    except:
        # If an error occurs (e.g., infeasible constraints), return None
        return None

Now, let's find the optimal portfolios and plot the efficient frontier.

In [ ]:
# Convert mean_returns and cov_matrix to numpy arrays for optimization
mean_returns_array = mean_returns.values
cov_matrix_array = cov_matrix.values

# Find the minimum volatility portfolio
min_vol_portfolio = get_min_volatility_portfolio(mean_returns_array, cov_matrix_array)

# Find the maximum Sharpe ratio portfolio
max_sharpe_portfolio = get_max_sharpe_portfolio(mean_returns_array, cov_matrix_array)

# Calculate the efficient frontier
returns_range = np.linspace(min_vol_portfolio['return'], np.max(mean_returns_array), 50)
efficient_portfolios = get_efficient_frontier(mean_returns_array, cov_matrix_array, returns_range)

# Filter out None values (infeasible portfolios)
efficient_portfolios = [p for p in efficient_portfolios if p is not None]

In [ ]:
# Extract volatility and return values from efficient portfolios
efficient_volatilities = [p['volatility'] for p in efficient_portfolios]
efficient_returns = [p['return'] for p in efficient_portfolios]

# Plot the efficient frontier
plt.figure(figsize=(14, 8))

# Plot individual sector ETFs
plt.scatter(volatility, mean_returns, s=100, alpha=0.5, label='Sector ETFs')

# Add labels for each sector
for i, ticker in enumerate(risk_return_metrics.index):
    plt.annotate(ticker, (volatility[i], mean_returns[i]), 
                 textcoords="offset points", xytext=(5,5), ha='left')

# Plot minimum volatility portfolio
plt.scatter(min_vol_portfolio['volatility'], min_vol_portfolio['return'], 
            s=150, color='g', marker='*', label='Minimum Volatility Portfolio')

# Plot maximum Sharpe ratio portfolio
plt.scatter(max_sharpe_portfolio['volatility'], max_sharpe_portfolio['return'], 
            s=150, color='r', marker='*', label='Maximum Sharpe Ratio Portfolio')

# Plot efficient frontier
plt.plot(efficient_volatilities, efficient_returns, 'b--', linewidth=3, label='Efficient Frontier')

plt.title('Efficient Frontier of S&P 500 Sector ETFs')
plt.xlabel('Annual Volatility (Risk)')
plt.ylabel('Annual Return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Let's analyze the composition of the optimal portfolios.

In [ ]:
# Create functions to display portfolio compositions
def display_portfolio_composition(portfolio, title, tickers, sector_names):
    """Display the composition of a portfolio with a pie chart"""
    # Create a DataFrame with weights
    composition = pd.DataFrame({
        'Ticker': tickers,
        'Sector': [sector_names[ticker] for ticker in tickers],
        'Weight': portfolio['weights'] * 100  # Convert to percentage
    })
    
    # Filter out sectors with very small allocations for clarity
    significant_allocation = composition[composition['Weight'] > 1].copy()
    significant_allocation.sort_values('Weight', ascending=False, inplace=True)
    
    # Display tabular results
    print(f"\n{title}:")
    print(f"Annual Return: {portfolio['return']*100:.2f}%")
    print(f"Annual Volatility: {portfolio['volatility']*100:.2f}%")
    print(f"Sharpe Ratio: {portfolio['sharpe']:.2f}")
    print("\nPortfolio Composition:")
    display(significant_allocation.sort_values('Weight', ascending=False))
    
    # Create pie chart of weights
    plt.figure(figsize=(12, 8))
    plt.pie(significant_allocation['Weight'], labels=[f"{row['Ticker']} ({row['Sector']})" for _, row in significant_allocation.iterrows()], 
            autopct='%1.1f%%', startangle=90, shadow=True)
    plt.title(f"{title} Composition")
    plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
    plt.tight_layout()
    plt.show()
    
    return significant_allocation

In [ ]:
# Display minimum volatility portfolio
min_vol_composition = display_portfolio_composition(
    min_vol_portfolio, 
    "Minimum Volatility Portfolio", 
    returns.columns, 
    sectors
)

In [ ]:
# Display maximum Sharpe ratio portfolio
max_sharpe_composition = display_portfolio_composition(
    max_sharpe_portfolio, 
    "Maximum Sharpe Ratio Portfolio", 
    returns.columns, 
    sectors
)

## Portfolio Rebalancing Framework

In algorithmic trading, we often need to periodically rebalance our portfolio to maintain optimal allocation. Let's implement a simple rebalancing framework.

In [ ]:
def backtest_portfolio_rebalancing(prices, rebalance_frequency='M', optimization_function=get_max_sharpe_portfolio):
    """Backtest a portfolio with periodic rebalancing"""
    # Convert to pandas DatetimeIndex for resampling
    prices.index = pd.DatetimeIndex(prices.index)
    
    # Get rebalance dates
    rebalance_dates = prices.resample(rebalance_frequency).last().index
    rebalance_dates = rebalance_dates.intersection(prices.index)
    
    # Initialize weights with equal allocation
    current_weights = np.ones(len(prices.columns)) / len(prices.columns)
    
    # Track portfolio value and allocations
    portfolio_value = 100  # Starting with $100
    portfolio_values = [portfolio_value]
    weight_history = [current_weights]
    dates = [prices.index[0]]
    
    # Initialize lookback period (in days)
    lookback_days = 252  # Approximately 1 year of trading days
    
    # Loop through the price history
    for i in range(1, len(prices)):
        current_date = prices.index[i]
        previous_date = prices.index[i-1]
        
        # Calculate daily returns
        daily_returns = prices.loc[current_date] / prices.loc[previous_date] - 1
        
        # Update portfolio value
        portfolio_value *= (1 + np.sum(daily_returns * current_weights))
        
        # Check if this is a rebalance date
        if current_date in rebalance_dates and i >= lookback_days:
            # Get historical data for optimization (lookback period)
            historical_data = prices.iloc[i-lookback_days:i]
            historical_returns = historical_data.pct_change().dropna()
            
            # Calculate mean returns and covariance matrix
            mean_returns = historical_returns.mean() * 252
            cov_matrix = historical_returns.cov() * 252
            
            # Optimize portfolio weights
            optimal_portfolio = optimization_function(mean_returns.values, cov_matrix.values)
            current_weights = optimal_portfolio['weights']
            
            # Log rebalancing
            print(f"Rebalanced on {current_date.date()} - Portfolio Value: ${portfolio_value:.2f}")
        
        # Track portfolio value and weights
        portfolio_values.append(portfolio_value)
        weight_history.append(current_weights)
        dates.append(current_date)
    
    # Create a DataFrame with portfolio values
    portfolio_df = pd.DataFrame({
        'Date': dates,
        'Portfolio Value': portfolio_values
    }).set_index('Date')
    
    # Create a DataFrame with weight history
    weights_df = pd.DataFrame(
        weight_history, 
        columns=prices.columns,
        index=dates
    )
    
    return portfolio_df, weights_df

In [ ]:
# Backtest maximum Sharpe ratio portfolio with quarterly rebalancing
sharpe_portfolio_values, sharpe_weights = backtest_portfolio_rebalancing(
    prices,
    rebalance_frequency='Q',  # Quarterly
    optimization_function=get_max_sharpe_portfolio
)

In [ ]:
# Backtest minimum volatility portfolio with quarterly rebalancing
min_vol_portfolio_values, min_vol_weights = backtest_portfolio_rebalancing(
    prices,
    rebalance_frequency='Q',  # Quarterly
    optimization_function=get_min_volatility_portfolio
)

In [ ]:
# Calculate an equal-weight portfolio as benchmark
def equal_weight_portfolio(mean_returns, cov_matrix):
    """Create an equal-weight portfolio"""
    num_assets = len(mean_returns)
    weights = np.ones(num_assets) / num_assets
    portfolio_return, portfolio_volatility = portfolio_performance(weights, mean_returns, cov_matrix)
    sharpe_ratio = portfolio_sharpe_ratio(weights, mean_returns, cov_matrix)
    
    return {
        'weights': weights,
        'return': portfolio_return,
        'volatility': portfolio_volatility,
        'sharpe': sharpe_ratio
    }

# Backtest equal-weight portfolio
equal_portfolio_values, equal_weights = backtest_portfolio_rebalancing(
    prices,
    rebalance_frequency='Q',  # Quarterly
    optimization_function=equal_weight_portfolio
)

In [ ]:
# Also calculate S&P 500 index performance for comparison
sp500_index = yf.download('^GSPC', start=prices.index[0], end=prices.index[-1])['Adj Close']
sp500_index = sp500_index / sp500_index.iloc[0] * 100  # Normalize to 100

In [ ]:
# Compare the performance of different portfolios
plt.figure(figsize=(14, 8))
plt.plot(sharpe_portfolio_values, label='Max Sharpe Portfolio')
plt.plot(min_vol_portfolio_values, label='Min Volatility Portfolio')
plt.plot(equal_portfolio_values, label='Equal-Weight Portfolio')
plt.plot(sp500_index, label='S&P 500 Index')
plt.title('Portfolio Performance Comparison')
plt.xlabel('Date')
plt.ylabel('Portfolio Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate performance metrics for each portfolio
def calculate_performance_metrics(portfolio_values):
    """Calculate performance metrics for a portfolio value series"""
    # Calculate daily returns
    returns = portfolio_values.pct_change().dropna()
    
    # Calculate metrics
    total_return = (portfolio_values.iloc[-1] / portfolio_values.iloc[0]) - 1
    annualized_return = (1 + total_return) ** (252 / len(returns)) - 1
    annualized_volatility = returns.std() * np.sqrt(252)
    sharpe_ratio = annualized_return / annualized_volatility
    max_drawdown = (portfolio_values / portfolio_values.cummax() - 1).min()
    
    return {
        'Total Return': total_return,
        'Annualized Return': annualized_return,
        'Annualized Volatility': annualized_volatility,
        'Sharpe Ratio': sharpe_ratio,
        'Max Drawdown': max_drawdown
    }

# Calculate metrics for each portfolio
sharpe_metrics = calculate_performance_metrics(sharpe_portfolio_values['Portfolio Value'])
min_vol_metrics = calculate_performance_metrics(min_vol_portfolio_values['Portfolio Value'])
equal_metrics = calculate_performance_metrics(equal_portfolio_values['Portfolio Value'])
sp500_metrics = calculate_performance_metrics(sp500_index)

# Create a DataFrame to compare metrics
metrics_comparison = pd.DataFrame({
    'Max Sharpe Portfolio': sharpe_metrics,
    'Min Volatility Portfolio': min_vol_metrics,
    'Equal-Weight Portfolio': equal_metrics,
    'S&P 500 Index': sp500_metrics
})

# Format the metrics
formatted_metrics = metrics_comparison.copy()
for metric in ['Total Return', 'Annualized Return', 'Annualized Volatility', 'Max Drawdown']:
    formatted_metrics.loc[metric] = formatted_metrics.loc[metric].apply(lambda x: f"{x*100:.2f}%")
formatted_metrics.loc['Sharpe Ratio'] = formatted_metrics.loc['Sharpe Ratio'].apply(lambda x: f"{x:.2f}")

# Display metrics comparison
print("Portfolio Performance Comparison:")
display(formatted_metrics)

## Visualizing Sector Allocation Over Time

Let's visualize how the sector allocations changed over time with our rebalancing strategy.

In [ ]:
# Function to plot allocation over time
def plot_allocation_over_time(weights_df, title):
    """Plot the allocation of assets over time"""
    # Resample to monthly frequency for better visualization
    monthly_weights = weights_df.resample('M').last()
    
    # Plot the allocation
    plt.figure(figsize=(14, 8))
    ax = monthly_weights.plot.area(figsize=(14, 8), alpha=0.8, title=title)
    plt.xlabel('Date')
    plt.ylabel('Allocation')
    
    # Add legend with sector names
    handles, labels = ax.get_legend_handles_labels()
    ax.legend([f"{ticker} ({sectors[ticker]})" for ticker in monthly_weights.columns], 
              loc='center left', bbox_to_anchor=(1, 0.5))
    
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot allocation over time for maximum Sharpe ratio portfolio
plot_allocation_over_time(sharpe_weights, 'Max Sharpe Portfolio Allocation Over Time')

In [ ]:
# Plot allocation over time for minimum volatility portfolio
plot_allocation_over_time(min_vol_weights, 'Min Volatility Portfolio Allocation Over Time')

## Algorithmic Trading Strategy Implementation

Now, let's implement a simple algorithmic trading strategy based on our portfolio optimization framework.

In [ ]:
def sector_rotation_strategy(prices, lookback_period=60, rebalance_frequency='M', volatility_target=0.15):
    """Implement a sector rotation strategy with volatility targeting"""
    # Initialize portfolio
    portfolio_value = 100
    portfolio_values = [portfolio_value]
    dates = [prices.index[0]]
    
    # Initial equal weights
    current_weights = np.ones(len(prices.columns)) / len(prices.columns)
    weights_history = [current_weights]
    
    # Convert to pandas DatetimeIndex for resampling
    prices.index = pd.DatetimeIndex(prices.index)
    
    # Get rebalance dates
    rebalance_dates = prices.resample(rebalance_frequency).last().index
    rebalance_dates = rebalance_dates.intersection(prices.index)
    
    # Track allocations
    allocations = []
    
    # Loop through price history (starting after lookback period)
    for i in range(1, len(prices)):
        current_date = prices.index[i]
        previous_date = prices.index[i-1]
        
        # Calculate daily returns
        daily_returns = prices.loc[current_date] / prices.loc[previous_date] - 1
        
        # Update portfolio value
        portfolio_value *= (1 + np.sum(daily_returns * current_weights))
        
        # Check if this is a rebalance date and we have enough history
        if current_date in rebalance_dates and i > lookback_period:
            # Get historical data for signal generation
            historical_data = prices.iloc[i-lookback_period:i]
            historical_returns = historical_data.pct_change().dropna()
            
            # Calculate momentum signals (last 3 months)
            momentum_window = min(63, len(historical_returns) - 1)  # ~3 months of trading days
            momentum = historical_data.iloc[-1] / historical_data.iloc[-momentum_window] - 1
            
            # Calculate volatility (annualized)
            volatility = historical_returns.std() * np.sqrt(252)
            
            # Rank sectors by momentum/volatility ratio (risk-adjusted momentum)
            risk_adj_momentum = momentum / volatility
            ranked_sectors = risk_adj_momentum.sort_values(ascending=False)
            
            # Select top half of sectors
            top_sectors = ranked_sectors.index[:len(ranked_sectors)//2]
            
            # Optimize allocation among top sectors
            top_sector_returns = historical_returns[top_sectors]
            mean_returns = top_sector_returns.mean() * 252
            cov_matrix = top_sector_returns.cov() * 252
            
            # Volatility targeting
            min_vol_portfolio = get_min_volatility_portfolio(mean_returns.values, cov_matrix.values)
            min_vol_weights = min_vol_portfolio['weights']
            min_vol_volatility = min_vol_portfolio['volatility']
            
            # Calculate scaling factor for volatility targeting
            scaling_factor = volatility_target / min_vol_volatility
            
            # Adjust weights for all sectors
            new_weights = np.zeros(len(prices.columns))
            for j, sector in enumerate(top_sectors):
                idx = prices.columns.get_loc(sector)
                new_weights[idx] = min_vol_weights[j] * scaling_factor
            
            # Allocate remaining to cash (just reduce overall allocation if scaling > 1)
            if scaling_factor < 1:
                cash_weight = 1 - scaling_factor
            else:
                cash_weight = 0
                
            # Normalize weights to ensure they sum to 1
            current_weights = new_weights / new_weights.sum() * (1 - cash_weight)
            
            # Log allocation
            allocation = pd.Series(current_weights, index=prices.columns)
            allocation['Cash'] = cash_weight
            allocations.append((current_date, allocation))
            
            # Log rebalancing
            print(f"Rebalanced on {current_date.date()} - Portfolio Value: ${portfolio_value:.2f}, Cash: {cash_weight*100:.2f}%")
        
        # Record portfolio value and weights
        portfolio_values.append(portfolio_value)
        weights_history.append(current_weights)
        dates.append(current_date)
    
    # Create DataFrame with portfolio values
    portfolio_df = pd.DataFrame({
        'Date': dates,
        'Portfolio Value': portfolio_values
    }).set_index('Date')
    
    # Create DataFrame with weights history
    weights_df = pd.DataFrame(
        weights_history, 
        columns=prices.columns,
        index=dates
    )
    
    # Create DataFrame with allocations
    allocations_df = pd.DataFrame([a[1] for a in allocations], index=[a[0] for a in allocations])
    
    return portfolio_df, weights_df, allocations_df

In [ ]:
# Run the sector rotation strategy
rotation_values, rotation_weights, rotation_allocations = sector_rotation_strategy(
    prices,
    lookback_period=60,  # 60 trading days (about 3 months)
    rebalance_frequency='M',  # Monthly rebalancing
    volatility_target=0.12  # 12% annualized volatility target
)

In [ ]:
# Compare performance with other strategies and the S&P 500
plt.figure(figsize=(14, 8))
plt.plot(rotation_values, label='Sector Rotation Strategy')
plt.plot(sharpe_portfolio_values, label='Max Sharpe Portfolio')
plt.plot(min_vol_portfolio_values, label='Min Volatility Portfolio')
plt.plot(sp500_index, label='S&P 500 Index')
plt.title('Strategy Performance Comparison')
plt.xlabel('Date')
plt.ylabel('Portfolio Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate performance metrics for sector rotation strategy
rotation_metrics = calculate_performance_metrics(rotation_values['Portfolio Value'])

# Add to comparison
metrics_comparison['Sector Rotation Strategy'] = rotation_metrics

# Format the metrics
formatted_metrics = metrics_comparison.copy()
for metric in ['Total Return', 'Annualized Return', 'Annualized Volatility', 'Max Drawdown']:
    formatted_metrics.loc[metric] = formatted_metrics.loc[metric].apply(lambda x: f"{x*100:.2f}%")
formatted_metrics.loc['Sharpe Ratio'] = formatted_metrics.loc['Sharpe Ratio'].apply(lambda x: f"{x:.2f}")

# Display metrics comparison
print("Strategy Performance Comparison:")
display(formatted_metrics)

In [ ]:
# Visualize sector allocation over time for the rotation strategy
# Include cash in the visualization
allocation_with_cash = rotation_weights.copy()
allocation_with_cash['Cash'] = 1 - rotation_weights.sum(axis=1)

# Plot allocation
plt.figure(figsize=(14, 8))
ax = allocation_with_cash.resample('M').last().plot.area(figsize=(14, 8), alpha=0.8, 
                                                     title='Sector Rotation Strategy Allocation Over Time')
plt.xlabel('Date')
plt.ylabel('Allocation')

# Add legend with sector names
handles, labels = ax.get_legend_handles_labels()
sector_labels = [f"{ticker} ({sectors[ticker]})" if ticker in sectors else ticker 
               for ticker in allocation_with_cash.columns]
ax.legend(sector_labels, loc='center left', bbox_to_anchor=(1, 0.5))

plt.grid(True)
plt.tight_layout()
plt.show()

## Conclusion and Trading Applications

In this notebook, we've demonstrated how to use financial mathematics for portfolio optimization of S&P 500 sector ETFs. We've covered:

1. Modern Portfolio Theory implementation to find efficient portfolios
2. Risk-return analysis of S&P 500 sectors
3. Portfolio rebalancing strategies
4. Sector rotation with volatility targeting

These techniques can be applied in algorithmic trading in several ways:

### Algorithmic Trading Applications

1. **ETF Basket Trading**
   - Use optimization to create baskets of sector ETFs
   - Automate periodic rebalancing based on changing market conditions
   
2. **Risk Management**
   - Implement volatility targeting to maintain consistent risk exposure
   - Dynamically adjust position sizes based on portfolio optimization
   
3. **Tactical Asset Allocation**
   - Rotate between sectors based on momentum, fundamental factors, or macroeconomic signals
   - Combine with mean-variance optimization for improved risk-adjusted returns
   
4. **Factor-Based Portfolio Construction**
   - Integrate factor exposures (value, momentum, quality) into the optimization process
   - Create multi-factor portfolios with optimal sector weights

### Enhancements and Extensions

To improve these strategies further, consider:

1. Incorporating transaction costs into the optimization
2. Using Black-Litterman model to blend views with market equilibrium
3. Implementing robust optimization to handle estimation errors
4. Adding constraints on sector exposures or factor loadings
5. Exploring alternative risk measures beyond variance (e.g., CVaR)
6. Integrating machine learning for return forecasting

These portfolio optimization techniques provide a solid mathematical foundation for systematic trading strategies in the S&P 500 universe.